# Demo No. 1: Hello binary!

These tutorials introduce the main ELISA objects. This first example constructs a binary system and inspects its derived properties and units. The second half explains initialization, automatic defaults, explicit overrides, nested spots and pulsations, and numerical snapshots.

**Recommended workflow:** prepare the final input parameters, construct fresh `Star` objects and their `BinarySystem` or `SingleSystem`, then use that initialized system for calculations. For a different physical model, prefer constructing a new system from your input descriptions. ELISA supports many edits followed by `init()`, but changing stars on the fly introduces an intermediate state in which inputs and derived properties disagree. The examples below explain that behavior so you can use it deliberately when needed.

Run the cells in order from a fresh kernel. These examples use bundled model data and geometry; they do not require external atmosphere or limb-darkening tables.

## Importing the model classes

A `BinarySystem` combines two `Star` instances with orbital parameters. Import both classes from the public `elisa` namespace:

In [1]:
import logging

from elisa import BinarySystem, Star

logging.getLogger("elisa").setLevel(logging.WARNING)

## Setting up the components

Define the two components first. When the binary system is initialized, omitted gravity-darkening exponents and bolometric albedos are interpolated from bundled temperature-dependent tables; metallicity defaults to `[M/H] = 0` (solar). The examples below also show explicit values. Here `gravity_darkening` is the exponent in the bolometric flux–gravity relation: before irradiation, local temperature scales as $T \propto g^{\beta/4}$.

`discretization_factor` controls the approximate angular spacing of the surface mesh, in degrees at input. It is not a fixed size for every triangle; a smaller value gives a finer mesh. ELISA may adjust an unspecified companion factor using the relative radii and temperatures.

`synchronicity` is the stellar spin angular frequency divided by the mean orbital angular frequency. A value of 1 is synchronous for a circular orbit, but not with the varying instantaneous angular speed of an eccentric orbit. `t_eff` sets the surface-averaged bolometric temperature scale, rather than the polar temperature. The bolometric albedo controls reprocessing of incident companion radiation; it is not an optical geometric albedo.

In [2]:
from astropy import units as u

primary = Star(
    mass=2.15 * u.solMass,
    surface_potential=3.6,
    synchronicity=1.0,
    t_eff=10000 * u.K,
    gravity_darkening=1.0,  # optional argument, otherwise interpolated
    discretization_factor=5,  # angular size (in degrees) of the surface elements, optional
    albedo=0.6,  # optional argument, otherwise interpolated
    metallicity=0.0,  # optional argument
)

secondary = Star(
    mass=0.45 * u.solMass,
    surface_potential=5.39,
    synchronicity=1.0,
    t_eff=6500 * u.K,
)

Component input attributes can be assigned directly. Assignment records the current value as an explicit input, but it does not rebuild dependent physical state. This example assigns the same synchronicity as the constructor; the system will be constructed below.

In [3]:
primary.synchronicity = 1.0

Read an attribute in the same way:

In [4]:
primary.synchronicity

1.0

Use these components to construct the binary system:

In [5]:
bs = BinarySystem(
    primary=primary,
    secondary=secondary,
    argument_of_periastron=58 * u.deg,
    gamma=-30.7 * u.km / u.s,
    period=2.5 * u.d,
    eccentricity=0.0,
    inclination=85 * u.deg,
    primary_minimum_time=2440000.00000 * u.d,
    phase_shift=0.0,
    distance=153 * u.pc,
)


System attributes can also be read and assigned. For example, change the eccentricity:

In [6]:
bs.eccentricity = 0.1
bs.eccentricity

0.1

After changing supported stellar or system inputs, call `bs.init()` before further calculations. It rebuilds both stars and the binary system, validates the inputs, and recomputes dependent properties such as the orbit, reference radii and morphology. Assignment alone does not perform those calculations.

Omitted temperature-dependent albedo and gravity darkening are recalculated from the current temperature. Explicit constructor values or later user assignments are preserved. Here the primary has explicit `albedo=0.6` and `gravity_darkening=1.0`, while the secondary uses automatic values. The examples later in this notebook demonstrate the distinction.

Use quantities when assigning dimensional values: bare assigned numbers use the stored internal units, which can differ from constructor input units. `init()` does not refresh numerical containers already created from the system, and it is not a general observer of arbitrary edits to internal arrays.

In [7]:
bs.init()

The secondary component has temperature-dependent defaults for albedo and gravity darkening:

In [8]:
bs.secondary.albedo, bs.secondary.gravity_darkening

(np.float64(0.7126008613327499), 0.5302088904333047)

In [9]:
bs.default_internal_units

## Accessing parameters

Initialization determines the semi-major axis, morphology and component radii. Convert the internal semi-major axis to solar radii:

In [10]:
bs.semi_major_axis * bs.default_internal_units.system.semi_major_axis.to(u.solRad)  # conversion to solar radii

np.float64(10.659024832175282)

Inspect the morphology:

In [11]:
bs.morphology

'detached'

Component radii are dimensionless lengths in units of the binary semi-major axis $a$. For this eccentric system they describe the reference surface at periastron. The polar radius lies along the rotation axis; side, backward and forward radii describe the orbital-plane directions perpendicular to, away from and toward the companion. The equivalent radius is $R_\mathrm{eq}=(3V/4\pi)^{1/3}$, using ELISA's numerical estimate of the reference volume.

In [12]:
print(f"Polar radius: {primary.polar_radius}")
print(f"Side radius: {primary.side_radius}")
print(f"Backward radius: {primary.backward_radius}")
print(f"Forward radius: {primary.forward_radius}")
print(f"Equivalent radius: {primary.equivalent_radius}")

Polar radius: 0.2959386513918747
Side radius: 0.3007760700336461
Backward radius: 0.3037820837135195
Forward radius: 0.30578783211662763
Equivalent radius: 0.30016327101059803


Inspect each component's critical surface potential:

In [13]:
print(f"Critical potential for primary component: {primary.critical_surface_potential}")
print(f"Critical potential for secondary component: {secondary.critical_surface_potential}")

Critical potential for primary component: 2.4269724166520685
Critical potential for secondary component: 2.440275836628986


Critical potentials are evaluated at periastron using the mass ratio and each component's synchronicity. The input surface potentials are dimensionless Roche potentials in ELISA's convention, not gravitational potentials in SI units. A detached component has a surface potential above its critical value. Classification at the critical surface uses a numerical tolerance; do not infer a precise Roche-lobe filling margin from rounded printed values. Circular synchronous binaries also support a common over-contact surface; eccentric or asynchronous configurations use component-specific critical conditions.

## Initializing a binary system from a dictionary

`BinarySystem.from_json()` accepts a parameter dictionary in either the `standard` or `community` format. Values may be numbers in the default input units or supported quantities expressed as strings. A JSON file can store the same dictionary after conversion to JSON-compatible values:

In [14]:
standard_params = {
    "system": {
        "inclination": 90.0,
        "period": 10.1,
        "argument_of_periastron": "90.0 deg",  # string representation of astropy quantity is also valid
        "gamma": 0.0,
        "eccentricity": 0.3,
        "primary_minimum_time": 0.0,
        "phase_shift": 0.0,
        "distance": "153 pc",
    },
    "primary": {
        "mass": "4.0e30 kg",
        "surface_potential": 7.1,
        "synchronicity": 1.0,
        "t_eff": 6500.0,
        "gravity_darkening": 1.0,
        "discretization_factor": 5,
        "albedo": 1.0,
        "metallicity": 0.0,
    },
    "secondary": {
        "mass": 2.0,  # default unit is solMass"
        "surface_potential": 7.1,
        "synchronicity": 1.0,
        "t_eff": 6500.0,
        "gravity_darkening": 1.0,
        "discretization_factor": 5,
        "albedo": 1.0,
        "metallicity": 0.0,
    },
}
standard_binary = BinarySystem.from_json(standard_params)

The `standard` format specifies both masses. The `community` format instead specifies the mass ratio $q=M_2/M_1$ and semi-major axis; together with the period, these determine the masses through Kepler's law. These two examples use different physical parameters and need not produce the same binary:

In [15]:
community_params = {
    "system": {
        "inclination": 90.0,
        "period": 10.1,
        "argument_of_periastron": 90.0,
        "gamma": 0.0,
        "eccentricity": 0.3,
        "primary_minimum_time": 0.0,
        "phase_shift": 0.0,
        "semi_major_axis": 10.5,  # default unit is solRad
        "mass_ratio": 0.5,
        "distance": "153 pc",
    },
    "primary": {
        "surface_potential": 7.1,
        "synchronicity": 1.0,
        "t_eff": "6500.0 K",
        "gravity_darkening": 1.0,
        "discretization_factor": 5,
        "albedo": 1.0,
        "metallicity": 0.0,
    },
    "secondary": {
        "surface_potential": 7.1,
        "synchronicity": 1.0,
        "t_eff": 6500.0,
        "gravity_darkening": 1.0,
        "discretization_factor": 5,
        "albedo": 1.0,
        "metallicity": 0.0,
    },
}

community_binary = BinarySystem.from_json(community_params)

## Internal and input units

ELISA converts input values to its internal units. These are not uniformly SI: for example, distance is in metres but the orbital period is in days. Inspect units rather than assuming a convention:

In [16]:
from elisa import units

units.DISTANCE_UNIT, units.PERIOD_UNIT

(Unit("m"), Unit("d"))

Access the internal unit of an individual parameter:

In [17]:
bs.default_internal_units.system.inclination, bs.default_internal_units.primary.mass

(Unit("rad"), Unit("kg"))

Inspect the complete internal-unit mapping:

In [18]:
bs.default_internal_units.as_dict()

{'system': {'inclination': 'rad',
  'period': 'd',
  'eccentricity': '',
  'argument_of_periastron': 'rad',
  'gamma': 'm / s',
  'phase_shift': '',
  'additional_light': '',
  'primary_minimum_time': 'd',
  'semi_major_axis': 'm',
  'distance': 'm',
  'mass_ratio': ''},
 'component': {'mass': 'kg',
  't_eff': 'K',
  'surface_potential': '',
  'synchronicity': '',
  'metallicity': '',
  'gravity_darkening': '',
  'albedo': '',
  'discretization_factor': 'rad',
  'limb_darkening_coefficients': '',
  'spots': {'longitude': 'rad',
   'latitude': 'rad',
   'angular_radius': 'rad',
   'temperature_factor': '',
   'discretization_factor': 'rad'},
  'pulsations': {'l': '',
   'm': '',
   'amplitude': 'm / s',
   'frequency': 'Hz',
   'start_phase': 'rad',
   'mode_axis_theta': 'rad',
   'mode_axis_phi': 'rad',
   'temperature_perturbation_phase_shift': 'rad',
   'horizontal_to_radial_amplitude_ratio': '',
   'temperature_amplitude_factor': '',
   'tidally_locked': "<class 'bool'>"}},
 'primar

Bare numbers passed to constructors use the default input units, which can differ from the internal units. For example, inclination is entered in degrees but stored in radians, and stellar mass is entered in solar masses but stored in kilograms:

In [19]:
bs.default_input_units.system.inclination, bs.default_input_units.primary.mass

(Unit("deg"), Unit("solMass"))

Inspect all default input units:

In [20]:
bs.default_input_units.as_dict()

{'system': {'inclination': 'deg',
  'period': 'd',
  'eccentricity': '',
  'argument_of_periastron': 'deg',
  'gamma': 'm / s',
  'phase_shift': '',
  'additional_light': '',
  'primary_minimum_time': 'd',
  'semi_major_axis': 'solRad',
  'distance': 'pc',
  'mass_ratio': ''},
 'component': {'mass': 'solMass',
  't_eff': 'K',
  'surface_potential': '',
  'synchronicity': '',
  'metallicity': '',
  'gravity_darkening': '',
  'albedo': '',
  'discretization_factor': 'deg',
  'limb_darkening_coefficients': '',
  'spots': {'longitude': 'deg',
   'latitude': 'deg',
   'angular_radius': 'deg',
   'temperature_factor': '',
   'discretization_factor': 'deg'},
  'pulsations': {'l': '',
   'm': '',
   'amplitude': 'm / s',
   'frequency': '1 / d',
   'start_phase': 'rad',
   'mode_axis_theta': 'deg',
   'mode_axis_phi': 'deg',
   'temperature_perturbation_phase_shift': 'rad',
   'horizontal_to_radial_amplitude_ratio': '',
   'temperature_amplitude_factor': '',
   'tidally_locked': "<class 'bool'

Or inspect only the system input units:

In [21]:
bs.default_input_units.system.as_dict()

{'inclination': 'deg',
 'period': 'd',
 'eccentricity': '',
 'argument_of_periastron': 'deg',
 'gamma': 'm / s',
 'phase_shift': '',
 'additional_light': '',
 'primary_minimum_time': 'd',
 'semi_major_axis': 'solRad',
 'distance': 'pc',
 'mass_ratio': ''}

## Working with a model after initialization

The simplest workflow is to build a new system for each set of physical inputs. Let's use a small helper so we can explore several configurations without accidentally sharing stars between them. Each call creates fresh components and returns a ready-to-use binary.

We leave out `albedo` and `gravity_darkening` unless we want to choose their values ourselves. Compare the two primary temperatures below: the system chooses different defaults for the cool and hot star.

In [22]:
import pandas as pd


def make_example_binary(temperature=6000 * u.K, **primary_overrides):
    primary_inputs = {
        "mass": 1.2 * u.solMass,
        "t_eff": temperature,
        "surface_potential": 6.0,
        "synchronicity": 1.0,
    }
    primary_inputs.update(primary_overrides)
    return BinarySystem(
        primary=Star(**primary_inputs),
        secondary=Star(
            mass=0.8 * u.solMass, t_eff=5000 * u.K,
            surface_potential=6.0, synchronicity=1.0,
        ),
        inclination=80 * u.deg,
        period=3 * u.d,
        eccentricity=0.1,
        argument_of_periastron=45 * u.deg,
    )


cool_binary = make_example_binary()
hot_binary = make_example_binary(8500 * u.K)

pd.DataFrame({
    "Temperature [K]": [cool_binary.primary.t_eff, hot_binary.primary.t_eff],
    "Albedo": [cool_binary.primary.albedo, hot_binary.primary.albedo],
    "Gravity darkening": [cool_binary.primary.gravity_darkening, hot_binary.primary.gravity_darkening],
}, index=["Cool primary", "Hot primary"]).round(4)

,Temperature [K],Albedo,Gravity darkening
Cool primary,6000.0,0.6405,0.2434
Hot primary,8500.0,0.9728,0.9990


### Automatic values follow the current inputs after `init()`

Let's take the cool binary and raise its primary temperature to 8500 K. Assigning `t_eff` changes that input immediately, but albedo and gravity darkening still describe the old temperature until we call `init()`.

The table shows both stages. After initialization, the defaults match the hot model above. Reading an automatic value leaves it automatic; ELISA recalculates it whenever the relevant inputs change and the system is initialized again.

Call `binary.init()` for this operation. `star.init()` alone does not perform the system's interpolation or update the orbit and other dependent properties. Repeating `binary.init()` without further edits keeps the same physical model.

In [23]:
edited_binary = make_example_binary()

# Changing the temperature alone leaves the old defaults in place.
edited_binary.primary.t_eff = 8500 * u.K
before_init = {
    "Temperature [K]": edited_binary.primary.t_eff,
    "Albedo": edited_binary.primary.albedo,
    "Gravity darkening": edited_binary.primary.gravity_darkening,
}

# Rebuild the complete system to calculate the new defaults.
edited_binary.init()
after_init = {
    "Temperature [K]": edited_binary.primary.t_eff,
    "Albedo": edited_binary.primary.albedo,
    "Gravity darkening": edited_binary.primary.gravity_darkening,
}

pd.DataFrame([before_init, after_init], index=["After assignment", "After init()"]).round(4)

,Temperature [K],Albedo,Gravity darkening
After assignment,8500.0 K,0.6405,0.2434
After init(),8500.0,0.9728,0.9990


### Explicit overrides remain explicit

Supplying `albedo=0.7` at construction keeps that value through subsequent temperature changes and `init()` calls. A later assignment has the same effect. Even assigning the *same* value that ELISA calculated turns that parameter into an explicit override. Reading the value is safe; writing it back changes its meaning.

There is no public “return this field to automatic” operation. To return to automatic behavior, construct a fresh component and system while omitting that argument. Do not use `None`, `NaN`, or edits to tracking metadata as a general reset mechanism.

In [24]:
overridden = make_example_binary(albedo=0.7, gravity_darkening=0.4)
overridden.primary.t_eff = 8500 * u.K
overridden.init()

pinned = make_example_binary()
pinned.primary.albedo = pinned.primary.albedo  # Make the current automatic value explicit.
pinned.primary.t_eff = 8500 * u.K
pinned.init()

pd.DataFrame({
    "Albedo": [
        hot_binary.primary.albedo,
        overridden.primary.albedo,
        pinned.primary.albedo,
    ],
    "Gravity darkening": [
        hot_binary.primary.gravity_darkening,
        overridden.primary.gravity_darkening,
        pinned.primary.gravity_darkening,
    ],
}, index=[
    "Automatic at 8500 K",
    "Explicit values from the constructor",
    "Albedo pinned at its 6000 K value",
]).round(4)

,Albedo,Gravity darkening
Automatic at 8500 K,0.9728,0.999
Explicit values from the constructor,0.7000,0.400
Albedo pinned at its 6000 K value,0.6405,0.999


### Units and validation during edits

Constructors interpret bare numbers in default **input** units; existing attributes hold **internal** values. For example, constructor mass defaults to solar masses, whereas a bare mass assigned to an existing star is interpreted as kilograms when it is rebuilt. Angles are usually stored in radians. For `polar_log_g`, a bare constructor value follows the cgs convention, whereas the internal logarithm uses acceleration in m/s².

Prefer explicit quantities for dimensional assignments, then call the parent `init()` before reading derived state. The serializers attach internal units to stored numbers so repeated initialization does not repeatedly convert kg as solar masses or radians as degrees.

Many scalar assignments are only validated on reconstruction. If `init()` raises, do not continue synthesis with that model: rebuilding is not transactional, and some state may already have changed. Correct the inputs and initialize again, or construct a fresh model from your saved input description.

In [25]:
unit_example = make_example_binary()
unit_example.primary.mass = 1.3 * u.solMass
unit_example.inclination = 82 * u.deg
unit_example.init()

pd.DataFrame({
    "Stored value": [unit_example.primary.mass, unit_example.inclination],
    "Internal unit": ["kg", "rad"],
    "Converted back": [
        (unit_example.primary.mass * u.kg).to_value(u.solMass),
        (unit_example.inclination * u.rad).to_value(u.deg),
    ],
    "Display unit": ["solar masses", "degrees"],
}, index=["Primary mass", "Inclination"])

,Stored value,Internal unit,Converted back,Display unit
Primary mass,2.584933e+30,kg,1.3,solar masses
Inclination,1.431170e+00,rad,82.0,degrees


### Editing spots and pulsations

A spot and a pulsation mode can be supplied as descriptor dictionaries when creating a star. Here the spot inherits its mesh step from the star, and ELISA calculates the mode's horizontal-to-radial amplitude ratio.

We then move the spot, make it cooler, change the stellar mesh step and raise the stellar mass. After `nested_binary.init()`, the spot reflects our edits and the mode has a newly calculated amplitude ratio. The variables `spot` and `mode` still refer to the same objects: initialization updates existing nested objects rather than replacing them.

For modes, `amplitude` is a velocity amplitude; the derived `radial_amplitude` is a displacement. The system recalculates displacement amplitudes and omitted amplitude factors from the current stellar parameters. If you supply `horizontal_to_radial_amplitude_ratio` or `temperature_amplitude_factor` explicitly, ELISA preserves your chosen factor instead.

Initialize the complete system after nested edits. Calling only `mode.init()` clears its derived amplitudes and synthesis arrays without recalculating the parent-dependent values. Spot mesh steps also remain subject to the existing mesh-size limits, including the spot's angular radius.

In [26]:
nested_binary = make_example_binary(
    discretization_factor=5 * u.deg,
    spots=[dict(longitude=45, latitude=60, angular_radius=20, temperature_factor=0.9)],
    pulsations=[dict(l=1, m=1, amplitude=1 * u.m / u.s, frequency=12 / u.d)],
)
spot = nested_binary.primary.spots[0]
mode = nested_binary.primary.pulsations[0]

before_edit = {
    "Spot longitude [deg]": (spot.longitude * u.rad).to_value(u.deg),
    "Spot temperature factor": spot.temperature_factor,
    "Spot mesh step [deg]": (spot.discretization_factor * u.rad).to_value(u.deg),
    "Horizontal/radial amplitude ratio": mode.horizontal_to_radial_amplitude_ratio,
}

spot.longitude = 75 * u.deg
spot.temperature_factor = 0.8
nested_binary.primary.discretization_factor = 6 * u.deg
nested_binary.primary.mass = 1.4 * u.solMass
nested_binary.init()

after_edit = {
    "Spot longitude [deg]": (spot.longitude * u.rad).to_value(u.deg),
    "Spot temperature factor": spot.temperature_factor,
    "Spot mesh step [deg]": (spot.discretization_factor * u.rad).to_value(u.deg),
    "Horizontal/radial amplitude ratio": mode.horizontal_to_radial_amplitude_ratio,
}

pd.DataFrame({"Original model": before_edit, "After edits and init()": after_edit}).round(4)

,Original model,After edits and init()
Spot longitude [deg],45.0000,75.0000
Spot temperature factor,0.9000,0.8000
Spot mesh step [deg],5.0000,6.0000
Horizontal/radial amplitude ratio,0.0942,0.1063


The collection setters accept lists of descriptor dictionaries. Replacing `star.spots` or `star.pulsations` creates new nested objects; identity preservation applies to `init()`, not collection replacement. Replace a collection through its setter to change membership. Avoid manually inserting or deleting dictionary keys: numerical code expects contiguous indices starting at zero, and pulsation synthesis accesses mode `0`. `init()` does not repair gaps in those indices.

For example, `star.pulsations = [dict(l=1, m=1, amplitude=1, frequency=12)]` replaces the modes; assigning `star.pulsations = []` removes them. Initialize the complete system after changing its collections. See demos 4 and 5 for geometry and synthesis examples.

### A single star has a different surface description

For an isolated star, start with `mass`, `t_eff` and `polar_log_g`, then supply inclination and rotation period to `SingleSystem`. The system derives the potential and radii; changing mass or polar gravity and calling `single.init()` recalculates that surface.

The stored `surface_potential` is negative and has units of m²/s². It is a different quantity from a binary component's dimensionless Roche potential. Do not pass it back as `surface_potential` to a `Star` constructor. If a star was originally given a positive Roche potential before being attached to a SingleSystem, SingleSystem replaces it with its own derived potential and handles that distinction during reinitialization.

Automatic gravity darkening is refreshed for the single star as well. Binary irradiation albedo is not calculated for an isolated star.

In [27]:
from elisa import SingleSystem

single = SingleSystem(
    star=Star(
        mass=1 * u.solMass,
        t_eff=6000 * u.K,
        polar_log_g=4.4 * u.dex(u.cm / u.s ** 2),
    ),
    inclination=80 * u.deg,
    rotation_period=25 * u.d,
)

original_surface = {
    "Mass [solar masses]": (single.star.mass * u.kg).to_value(u.solMass),
    "Surface potential [m²/s²]": single.star.surface_potential,
    "Equivalent radius [solar radii]": (single.star.equivalent_radius * u.m).to_value(u.solRad),
}

single.star.mass = 1.2 * u.solMass
single.star.polar_log_g = 4.6 * u.dex(u.cm / u.s ** 2)
single.init()

updated_surface = {
    "Mass [solar masses]": (single.star.mass * u.kg).to_value(u.solMass),
    "Surface potential [m²/s²]": single.star.surface_potential,
    "Equivalent radius [solar radii]": (single.star.equivalent_radius * u.m).to_value(u.solRad),
}

pd.DataFrame([original_surface, updated_surface], index=["Original star", "After mass/gravity edit"])

,Mass [solar masses],Surface potential [m²/s²],Equivalent radius [solar radii]
Original star,1.0,-1.825781e+11,1.044793
After mass/gravity edit,1.2,-2.517905e+11,0.909116


### Current state and reconstruction inputs are different

| Method | What it provides | Typical purpose |
| --- | --- | --- |
| `kwargs_serializer()` | Current constructor fields, with physical units where needed, including calculated defaults and nested descriptors | Inspect or serialize current values |
| `reinitialization_kwargs()` | Current explicit and required inputs, omitting automatic optional defaults | The reconstruction path used by `init()` |
| `properties_serializer()` | Current numerical properties, including derived values; Star also copies its spots and pulsation modes | Build numerical containers |

Full state is not the original input recipe. Reconstructing a model from a full descriptor can make formerly automatic values explicit. JSON exports describe current state and do not preserve the automatic/explicit distinction. Keep your original input dictionaries if you need to retain that intent.

`kwargs_serializer()` is also not a universal cloning API: a system's components are supplied separately, some optional sentinels have special handling, and a SingleSystem star's derived negative potential is not a valid Roche constructor input.

In [28]:
automatic = make_example_binary()
current_values = automatic.primary.kwargs_serializer()
initialization_inputs = automatic.primary.reinitialization_kwargs()

pd.DataFrame({
    "Current value": [current_values["mass"], current_values["albedo"]],
    "Used as an explicit input by init()": [
        "mass" in initialization_inputs,
        "albedo" in initialization_inputs,
    ],
}, index=["Mass", "Albedo"])

,Current value,Used as an explicit input by init()
Mass,2.386091844837661e+30 kg,True
Albedo,0.640533,False


### A surface container keeps the model it was built from

`build_container()` builds a numerical surface at the requested phase. The resulting container owns its own spot and pulsation objects and the arrays needed for calculations.

Here we build a surface, change the model mass, and build another surface. The first container still holds the original mass and pulsation amplitude. This is intentional: it is a snapshot of the earlier model, not a live view of the updated star. Its existing geometry and pulsation arrays stay with that earlier configuration too.

Build a new container when you want to use the updated model. The normal light-curve, radial-velocity and plotting entry points prepare containers for their calls, so this mainly matters when you keep and work with containers yourself. Binary position containers follow the same rule.

In [29]:
snapshot_system = SingleSystem(
    star=Star(
        mass=1 * u.solMass, t_eff=6000 * u.K,
        polar_log_g=4.4 * u.dex(u.cm / u.s ** 2),
        pulsations=[dict(l=1, m=1, amplitude=1 * u.m / u.s, frequency=12 / u.d)],
    ),
    inclination=80 * u.deg,
    rotation_period=25 * u.d,
)
old_container = snapshot_system.build_container(phase=0.0)

snapshot_system.star.mass = 1.2 * u.solMass
snapshot_system.init()
new_container = snapshot_system.build_container(phase=0.0)

pd.DataFrame({
    "Mass [solar masses]": [
        (old_container.star.mass * u.kg).to_value(u.solMass),
        (new_container.star.mass * u.kg).to_value(u.solMass),
    ],
    "Radial displacement amplitude [m]": [
        old_container.star.pulsations[0].radial_amplitude,
        new_container.star.pulsations[0].radial_amplitude,
    ],
}, index=["Container built before the change", "Container built after the change"]).round(4)

,Mass [solar masses],Radial displacement amplitude [m]
Container built before the change,1.0,964.4081
Container built after the change,1.2,988.7429


### Choosing how to work

For regular analysis, prepare input descriptions, initialize fresh components and their system, then compute geometry, light curves or radial velocities. Keep the input description alongside the results so another model can be constructed without inheriting accidental overrides.

When an interactive edit is useful, change supported inputs with explicit units, initialize the complete parent system, and build new numerical containers for the new state. Remember that writing an automatic value back pins it, initializing a child does not initialize its parent, and a failed initialization may leave a partially rebuilt object.

ELISA tries to make supported edits and repeated initialization behave consistently. Prefer a stable initialized model during calculations rather than changing stellar state on the fly.

For implementation details and further examples, see [How parameter tracking and init() work](../src/elisa/base/README_PARAMETER_TRACKING.md).

## References:

- Claret, A. 2001, MNRAS. 327, 989–994
- Claret, A. 2003, A&A 406, 623–628